# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully referenceable by `@id` fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Use the API attributes (not as dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished if hasattr(dataset.metadata, 'datePublished') else 'N/A'}\nIdentifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review all available record sets, fields, and their `@id`s. This allows us to reference and manipulate the data precisely per Croissant best practices. The list below will enumerate the available top-level record sets and their fields/columns referenced by their `@id`.

In [ ]:
# List all record sets by @id
print("Available record sets (by @id):")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- {rs['@id'] if '@id' in rs else rs.id}: {rs['name'] if 'name' in rs else ''}")

if not record_sets:
    print("No explicit top-level record sets found. Listing all possible record sets from the dataset schema...")
    # Fallback: parse from metadata JSON-LD.
    meta_json = dataset.metadata.to_jsonld() if hasattr(dataset.metadata, 'to_jsonld') else dataset.metadata.to_json() if hasattr(dataset.metadata, 'to_json') else None
    # Try to find record sets by their type or property.
    rs_list = []
    if meta_json:
        # record sets are usually under 'recordSet' or '@graph' with 'RecordSet' @type
        # Try 'recordSet' field
        if 'recordSet' in meta_json:
            rs_list = meta_json['recordSet']
            for rs in rs_list:
                print(f"- {rs.get('@id', 'NO_ID')} {rs.get('name', '')}")
        elif '@graph' in meta_json:
            for entry in meta_json['@graph']:
                if entry.get('@type') == 'cr:RecordSet' or entry.get('@type') == 'RecordSet':
                    print(f"- {entry.get('@id', 'NO_ID')} {entry.get('name', '')}")

# For demonstration, let's programmatically collect all record set @ids
rs_ids = []
if record_sets:
    for rs in record_sets:
        if '@id' in rs:
            rs_ids.append(rs['@id'])
        elif hasattr(rs, 'id'):
            rs_ids.append(rs.id)
elif meta_json and 'recordSet' in meta_json:
    for rs in meta_json['recordSet']:
        if '@id' in rs:
            rs_ids.append(rs['@id'])
    
print("\nSummary of fields (by @id) in each record set:")
# For each record set, print field @id and name
for rs_id in rs_ids:
    recset = dataset.record_set(rs_id)
    print(f"\nRecord set: {rs_id}")
    fields = recset.fields
    for field in fields:
        # Support both dict and attribute
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, 'id', 'NO_ID')
        # Try to print the name/label/description if present
        field_name = field.get('name', field_id) if isinstance(field, dict) else getattr(field, 'name', field_id)
        print(f"    - {field_id}: {field_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
dataframes = {}

# For this dataset, usually there is a main record set with mass-spectrometry protein information.
if len(rs_ids) > 0:
    # We'll extract data from all loaded record sets
    for rs_id in rs_ids:
        print(f"Loading records from record set {rs_id} ...")
        # Each record is a dictionary. Reference by field @id.
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"[OK] Loaded {len(df)} records. Columns: {list(df.columns)}\n")
        else:
            print("[No records found]")
else:
    print("No record set IDs found in the metadata.")

# Show the columns from the main record set for exploration
if len(dataframes) > 0:
    # Just pick the first available record set
    first_rs_id = list(dataframes.keys())[0]
    print(f"Available columns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a specific numeric field, normalizing the field, removing outliers, and grouping by key attributes. All operations reference fields/columns by their `@id` as required.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# We'll use columns from the main record set for demonstration
if len(dataframes) > 0:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]

    # Inspect columns for numeric fields (by name or @id suffix)
    numeric_field_id = None
    group_field_id = None
    # Prioritize 'molecular_weight', 'coverage_percent', 'peptide_count', etc.
    for col in df.columns:
        # Common numeric fields by name
        if any(key in col.lower() for key in ['weight', 'coverage', 'count', 'abundance', 'mw', 'score', 'quant']):
            if np.issubdtype(df[col].dtype, np.number) or pd.to_numeric(df[col], errors='coerce').notna().sum() > 0:
                numeric_field_id = col
                break
    if numeric_field_id is None:
        # Else pick first float/integer-like field
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number) or pd.to_numeric(df[col], errors='coerce').notna().sum() > 0:
                numeric_field_id = col
                break

    # Pick a groupable field: e.g., 'description', 'accession', or anything string/categorical
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < min(10, len(df)//2) and df[col].dtype == object:
            group_field_id = col
            break

    print(f"Using record set id: {rs_id}")
    print(f"Numeric field for analysis (by @id): {numeric_field_id}")
    print(f"Group field for analysis (by @id): {group_field_id}")
    if numeric_field_id is not None:
        # Cast to numeric type
        field_vals = pd.to_numeric(df[numeric_field_id], errors='coerce')
        df[numeric_field_id] = field_vals
        threshold = field_vals.mean() if field_vals.mean() != np.nan else 0
        # Example: filter by > mean
        filtered_df = df[field_vals > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_field = f"{numeric_field_id}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_field]].head())

        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f'mean_{numeric_field_id}')
            print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No extracted dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we show the distribution of the numeric field and a boxplot by group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    field_vals = df[numeric_field_id].dropna()
    sns.histplot(field_vals, kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id available and has enough unique values
    if group_field_id:
        # Only plot if there are not too many groups
        if df[group_field_id].nunique() < 20:
            plt.figure(figsize=(10,5))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f'{numeric_field_id} by {group_field_id}')
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 mass spectrometry dataset using `mlcroissant`, dynamically referencing data entities by their `@id`. We demonstrated:
- How to discover record sets and fields (column) `@id`s.
- How to extract records into DataFrames and analyze their structure.
- Data processing steps such as filtering, normalization, and simple grouping.
- Visualization of numeric field distributions and group statistics.

This notebook provides a reproducible and precise workflow for working with FAIR datasets in the Croissant ecosystem.

_For more information, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/mlcroissant/)._